# Behavioral analysis: accuracy and RT by noise level

Correct-vs-incorrect responses and response time (RT) per noise level, CWS vs. CWNS, on the
`badaga` speech-in-noise task -- a first pass ahead of drift-diffusion modeling (DDM needs both
accuracy and RT per trial, so this notebook doubles as the sanity check for whether that data is
actually usable).

**Depends on a fix in `convert_behav_to_bids_WIP.ipynb`** (2026-08-07): the bulk events.tsv
conversion loop used to hardcode `response_key = np.nan` and pull `response_time` from the
scanner-trigger-wait component instead of the actual response component, so *no subject converted
through that path has usable accuracy, and RT was likely wrong too*. That's now fixed to match
the component the real behavioral response lives in (`Resp_Block_N.rt` / `Resp_Block_N.keys`) --
**events.tsv needs to be regenerated for every subject before this notebook's numbers mean
anything.** If accuracy looks like noise or RT distributions look implausible, check whether
you're looking at pre- or post-fix events.tsv first.

In [ ]:
import os
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.stats.anova import AnovaRM

In [ ]:
task_label = 'badaga'

bidsroot = os.path.join('/bgfs/bchandrasekaran/krs228/data/',
                        'SSP/',
                        'data_bids')
behavior_out_dir = os.path.join(bidsroot, 'derivatives', 'behavior')
os.makedirs(behavior_out_dir, exist_ok=True)

# cleanest -> noisiest, matching contrast_list's convention everywhere else in this repo
# (univariate_first-level.py, group_level_all_ROI.ipynb, GLMsingle_rsa-group.ipynb)
NOISE_LEVEL_ORDER = ['Q', '8', '0', 'n2', 'n6']

# same palette as group_level_all_ROI.ipynb's GROUP_PALETTE, kept identical so CWS/CWNS always
# map to the same colors across every notebook and the report
GROUP_ORDER = ['CWS', 'CWNS']
GROUP_PALETTE = {'CWNS': '#009E73', 'CWS': '#CC79A7'}  # bluish-green / reddish-purple

### Participants and group lookup

In [ ]:
participants_fpath = os.path.join(bidsroot, 'participants.tsv')
participants_df = pd.read_csv(participants_fpath, sep='\t')

# case-/whitespace-normalized group comparison -- same fix applied throughout this repo, since
# participants.tsv's group column has been observed with inconsistent casing (e.g. 'CWS' vs 'cws')
group_norm = participants_df.group.str.strip().str.lower()
group_lookup = {
    **{s: 'CWNS' for s in participants_df.participant_id[group_norm == 'control']},
    **{s: 'CWS' for s in participants_df.participant_id[group_norm == 'cws']},
}
sub_list = sorted(group_lookup.keys())
print(f'{sum(g == "CWNS" for g in group_lookup.values())} CWNS, '
     f'{sum(g == "CWS" for g in group_lookup.values())} CWS in participants.tsv')

### Load and pair sound/response trials per subject

Each run's events.tsv has one `'sound'` row (onset, syllable, speaker, noise_level) and one
`'response'` row (response_time, correct_key, response_key) per trial, written in that order by
the BIDS conversion notebook -- so pairing them back into one row per trial is a straightforward
positional zip within each run, not a fuzzy time-based match.

In [ ]:
def load_subject_events(sub_id, bidsroot, task_label):
    """Concatenate every run's events.tsv for one subject into a single dataframe, tagged with
    its run number (parsed from the filename, not assumed from list position -- a subject can be
    missing a run in the middle, e.g. the events/BOLD run-count mismatches investigated
    separately).
    """
    func_dir = os.path.join(bidsroot, sub_id, 'func')
    fpaths = sorted(glob(os.path.join(func_dir, f'{sub_id}_task-{task_label}_run-*_events.tsv')))
    run_dfs = []
    for fpath in fpaths:
        run_num = int(os.path.basename(fpath).split('run-')[1].split('_')[0])
        run_df = pd.read_csv(fpath, sep='\t')
        run_df['run'] = run_num
        run_dfs.append(run_df)
    if not run_dfs:
        return None
    return pd.concat(run_dfs, ignore_index=True)


def pair_trials(events_df):
    """One row per trial: sound-trial columns (syllable, speaker, noise_level) joined to that
    same trial's response columns (response_time, correct_key, response_key), paired positionally
    within each run (see markdown above -- conversion always writes one 'response' row
    immediately following its 'sound' row, in that order, per trial). Returns None if a run's
    sound/response counts don't match (flags a conversion problem rather than silently
    mispairing trials by truncating/guessing).
    """
    trial_dfs = []
    for run_num, run_df in events_df.groupby('run'):
        sound_df = run_df[run_df.trial_type == 'sound'].sort_values('onset').reset_index(drop=True)
        resp_df = run_df[run_df.trial_type == 'response'].sort_values('onset').reset_index(drop=True)
        if len(sound_df) != len(resp_df):
            print(f'  WARNING: run {run_num} has {len(sound_df)} sound rows but '
                 f'{len(resp_df)} response rows -- skipping this run (conversion problem, '
                 'not a pairing decision this function should make silently).')
            continue
        trial_df = pd.concat([
            sound_df[['onset', 'syllable', 'speaker', 'noise_level']],
            resp_df[['response_time', 'correct_key', 'response_key']],
        ], axis=1)
        trial_df['run'] = run_num
        trial_dfs.append(trial_df)
    if not trial_dfs:
        return None
    return pd.concat(trial_dfs, ignore_index=True)

### Build the full trial-level table

`responded` and `correct` are kept as separate columns rather than collapsing straight to
correct/incorrect -- a trial with no response at all (`response_key` is NaN) is not the same
thing as a trial where the participant pressed the wrong key, and DDM in particular needs that
distinction (omissions are usually excluded, not scored as errors).

In [ ]:
FORCE_RELOAD = False
trial_df_pkl_path = os.path.join(behavior_out_dir, 'trial_df_long.pkl')

if os.path.exists(trial_df_pkl_path) and not FORCE_RELOAD:
    print(f'Loading cached trial_df_long from {trial_df_pkl_path} (set FORCE_RELOAD=True to reload)')
    trial_df_long = pd.read_pickle(trial_df_pkl_path)
else:
    trial_rows = []
    for sub_id in sub_list:
        events_df = load_subject_events(sub_id, bidsroot, task_label)
        if events_df is None:
            print(f'No events.tsv found for {sub_id} -- skipping.')
            continue
        trial_df = pair_trials(events_df)
        if trial_df is None or len(trial_df) == 0:
            print(f'No usable trials for {sub_id} -- skipping.')
            continue
        trial_df['participant_id'] = sub_id
        trial_df['group'] = group_lookup[sub_id]
        trial_rows.append(trial_df)

    trial_df_long = pd.concat(trial_rows, ignore_index=True)
    trial_df_long['responded'] = trial_df_long.response_key.notna()
    trial_df_long['correct'] = trial_df_long.responded & (trial_df_long.response_key == trial_df_long.correct_key)
    trial_df_long['noise_level'] = pd.Categorical(
        trial_df_long['noise_level'], categories=NOISE_LEVEL_ORDER, ordered=True)

    trial_df_long.to_pickle(trial_df_pkl_path)

print(f'{len(trial_df_long)} trials across {trial_df_long.participant_id.nunique()} subjects')
print(f'Overall response rate: {trial_df_long.responded.mean():.1%}')
print(f'Overall accuracy (among responded trials): '
     f'{trial_df_long[trial_df_long.responded].correct.mean():.1%}')
trial_df_long.head()

### Sanity check: does `response_key` actually look usable?

Worth a look before trusting anything downstream -- if `response_key`/`correct_key` are on
different numeric ranges (e.g. keypad codes vs. a 1-4 condition index) rather than genuinely
comparable values, accuracy computed by direct equality above is meaningless regardless of
whether the events/BOLD conversion fix landed correctly.

In [ ]:
print('response_key values:', sorted(trial_df_long.response_key.dropna().unique()))
print('correct_key values:', sorted(trial_df_long.correct_key.dropna().unique()))
print(f'Response rate by subject -- min {trial_df_long.groupby("participant_id").responded.mean().min():.1%}, '
     f'max {trial_df_long.groupby("participant_id").responded.mean().max():.1%}')

### Per-subject, per-noise-level accuracy and RT

In [ ]:
def summarize_by_noise_level(trial_df_long):
    """One row per (subject, noise_level): response rate, accuracy (among responded trials),
    and mean RT for correct vs. all-responded trials.
    """
    rows = []
    for (sub_id, noise_level), g in trial_df_long.groupby(['participant_id', 'noise_level'], observed=True):
        responded = g[g.responded]
        rows.append({
            'participant_id': sub_id,
            'group': g['group'].iloc[0],
            'noise_level': noise_level,
            'n_trials': len(g),
            'response_rate': g.responded.mean(),
            'accuracy': responded.correct.mean() if len(responded) > 0 else np.nan,
            'rt_mean_correct': responded[responded.correct].response_time.mean() if responded.correct.any() else np.nan,
            'rt_mean_all_responded': responded.response_time.mean() if len(responded) > 0 else np.nan,
        })
    summary_df = pd.DataFrame(rows)
    summary_df['noise_level'] = pd.Categorical(
        summary_df['noise_level'], categories=NOISE_LEVEL_ORDER, ordered=True)
    return summary_df


behavior_summary_df = summarize_by_noise_level(trial_df_long)
behavior_summary_df.to_csv(os.path.join(behavior_out_dir, 'behavior_summary_by-noise-level.csv'), index=False)
behavior_summary_df.head()

### Plotting helpers (same box+strip convention as group_level_all_ROI.ipynb)

In [ ]:
def _outline_only(ax):
    """Strip fill from stripplot dots and boxplot boxes, re-applying each hue group's original
    fill color as the outline/edge color first -- matches group_level_all_ROI.ipynb's helper of
    the same name, duplicated here rather than imported since these notebooks don't share a
    module.
    """
    for collection in ax.collections:
        facecolor = collection.get_facecolor()
        collection.set_edgecolor(facecolor)
        collection.set_facecolor('none')
    for patch in ax.patches:
        facecolor = patch.get_facecolor()
        patch.set_edgecolor(facecolor)
        patch.set_facecolor('none')


def plot_behavior_by_noise_level(summary_df, value_col, ylabel, title):
    fig, ax = plt.subplots(1, 1, figsize=(0.9 * len(NOISE_LEVEL_ORDER) + 2, 4), dpi=300)

    sns.stripplot(data=summary_df, x='noise_level', y=value_col, hue='group',
                 order=NOISE_LEVEL_ORDER, hue_order=GROUP_ORDER, palette=GROUP_PALETTE,
                 dodge=True, linewidth=0.5, size=3, legend=None, ax=ax, zorder=2)
    sns.boxplot(data=summary_df, x='noise_level', y=value_col, hue='group',
               order=NOISE_LEVEL_ORDER, hue_order=GROUP_ORDER, palette=GROUP_PALETTE,
               dodge=True, linewidth=1, fliersize=0, ax=ax, zorder=1)
    _outline_only(ax)

    sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1), title='Group')
    ax.set_xlabel('noise level')
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    fig.tight_layout()
    sns.despine(ax=ax)
    return fig

### Accuracy by noise level

In [ ]:
fig = plot_behavior_by_noise_level(
    behavior_summary_df, 'accuracy', 'accuracy (among responded trials)',
    'Accuracy by noise level, CWS vs. CWNS')
fig.savefig(os.path.join(behavior_out_dir, 'accuracy_by-noise-level.png'), bbox_inches='tight')
fig.savefig(os.path.join(behavior_out_dir, 'accuracy_by-noise-level.svg'), bbox_inches='tight')

### RT by noise level (correct trials only)

In [ ]:
fig = plot_behavior_by_noise_level(
    behavior_summary_df, 'rt_mean_correct', 'mean RT, correct trials (s)',
    'Response time by noise level, CWS vs. CWNS')
fig.savefig(os.path.join(behavior_out_dir, 'rt_by-noise-level.png'), bbox_inches='tight')
fig.savefig(os.path.join(behavior_out_dir, 'rt_by-noise-level.svg'), bbox_inches='tight')

### Statistics: repeated-measures ANOVA (noise level x group)

Same balanced-panel requirement/gap as `group_level_all_ROI.ipynb`'s `restrict_to_complete_cases`
-- `AnovaRM` needs exactly one value per subject per noise level, and a subject missing a whole
noise level (e.g. from the RSA-motivated events/BOLD exclusions elsewhere in this pipeline) would
otherwise surface as an opaque "Data is unbalanced" error instead of a clear message about who's
missing and why.

In [ ]:
def restrict_to_complete_cases(long_df, factor, subject_col='participant_id'):
    cell_counts = long_df.groupby(subject_col, observed=True)[factor].nunique()
    n_levels = long_df[factor].nunique()
    incomplete = cell_counts[cell_counts != n_levels]
    if len(incomplete) > 0:
        print(f'Dropping {len(incomplete)} subject(s) with incomplete {factor} coverage '
             f'(needed for the balanced repeated-measures ANOVA): {list(incomplete.index)}')
    complete_subjects = cell_counts[cell_counts == n_levels].index
    return long_df[long_df[subject_col].isin(complete_subjects)]


for value_col, label in [('accuracy', 'Accuracy'), ('rt_mean_correct', 'RT (correct trials)')]:
    print(f'--- {label} ---')
    for group_name in ['CWNS', 'CWS']:
        group_df = behavior_summary_df[behavior_summary_df.group == group_name].dropna(subset=[value_col])
        group_df = restrict_to_complete_cases(group_df, 'noise_level')
        if group_df['participant_id'].nunique() < 2:
            print(f'  {group_name}: too few complete subjects for AnovaRM -- skipping.')
            continue
        aov = AnovaRM(group_df, depvar=value_col, subject='participant_id', within=['noise_level']).fit()
        print(f'  {group_name}:')
        print(aov.anova_table)
    print()

### Next steps

Once accuracy/RT look sane on real (post-fix) data: drift-diffusion modeling, approach TBD --
holding off on committing to a specific library (EZ-diffusion / HDDM / hssm) until this data has
actually been looked at.